In [25]:
# -*- coding: utf-8 -*-
"""Briscas AI with Monte Carlo sampling and corrected heuristic agent."""

import random
import copy
from collections import defaultdict

# ------------------------------
# Definición del juego (Briscas simplificado)
# ------------------------------
SUITS = ['oros', 'copas', 'espadas', 'bastos']
VALUES = [1,2,3,4,5,6,7,10,11,12]  # 1=as, 10=sota, 11=caballo, 12=rey
CARD_SCORES = {1:11, 3:10, 10:2, 11:3, 12:4}
for v in [2,4,5,6,7]:
    CARD_SCORES[v] = 0

def all_cards():
    return [(s,v) for s in SUITS for v in VALUES]

def card_score(card):
    return CARD_SCORES[card[1]]

class BriscasGame:
    def __init__(self, players=4):
        self.players = players
        self.team = [0,1,0,1]  # equipo 0: jugadores 0 y 2; equipo 1: jugadores 1 y 3

    def deal(self, seed=None):
        if seed:
            random.seed(seed)
        deck = all_cards()
        random.shuffle(deck)
        self.hands = [deck[i*3:(i+1)*3] for i in range(4)]  # 3 cartas por jugador
        self.trump = deck[-1][0]   # la última carta del mazo define el triunfo
        self.trick = {'leader':0, 'cards':[], 'suit':None, 'winner':None}
        self.current_player = 0
        self.scores = [0,0]  # puntos por equipo
        self.played_cards = []
        self.round_over = False

    def legal_actions(self, player):
        hand = self.hands[player]
        if self.trick['suit'] is None:
            return hand[:]
        else:
            suit = self.trick['suit']
            same_suit = [c for c in hand if c[0] == suit]
            if same_suit:
                return same_suit
            else:
                return hand

    def trick_winner(self, cards, trump):
        # cards: lista de tuplas (jugador, carta)
        lead_suit = cards[0][1][0]
        best = cards[0]
        for p,c in cards:
            if c[0] == trump and best[1][0] != trump:
                best = (p,c)
            elif c[0] == lead_suit and best[1][0] != trump:
                if c[1] > best[1][1] and best[1][0] == lead_suit:
                    best = (p,c)
            elif c[0] == trump and best[1][0] == trump:
                if c[1] > best[1][1]:
                    best = (p,c)
        return best[0]

    def play_card(self, player, card):
        self.hands[player].remove(card)
        self.trick['cards'].append((player, card))
        if len(self.trick['cards']) == 1:
            self.trick['suit'] = card[0]
        if len(self.trick['cards']) == 4:
            winner = self.trick_winner(self.trick['cards'], self.trump)
            trick_points = sum(card_score(c) for _,c in self.trick['cards'])
            team = self.team[winner]
            self.scores[team] += trick_points
            self.trick = {'leader':winner, 'cards':[], 'suit':None, 'winner':winner}
            self.current_player = winner
            if any(len(h)==0 for h in self.hands):
                self.round_over = True
        else:
            self.current_player = (self.current_player + 1) % 4
        self.played_cards.append(card)

    def is_terminal(self):
        return self.round_over or all(len(h)==0 for h in self.hands)

    def get_winner(self):
        if self.scores[0] > self.scores[1]:
            return 0
        elif self.scores[1] > self.scores[0]:
            return 1
        else:
            return -1

# ------------------------------
# Agente Monte Carlo (igual que antes)
# ------------------------------
class MonteCarloAgent:
    def __init__(self, env, player_id, num_samples=50):
        self.env = env
        self.player_id = player_id
        self.num_samples = num_samples

    def get_action(self, state):
        legal = state.legal_actions(self.player_id)
        if not legal:
            return None
        best_action = None
        best_value = -float('inf')
        for action in legal:
            win_count = 0
            for _ in range(self.num_samples):
                sim = copy.deepcopy(state)
                sim.play_card(self.player_id, action)
                self._random_playout(sim)
                winner = sim.get_winner()
                if winner == self.env.team[self.player_id]:
                    win_count += 1
            value = win_count / self.num_samples
            if value > best_value:
                best_value = value
                best_action = action
        return best_action

    def _random_playout(self, game):
        while not game.is_terminal():
            player = game.current_player
            legal = game.legal_actions(player)
            if legal:
                action = random.choice(legal)
                game.play_card(player, action)
            else:
                break

# ------------------------------
# Agente aleatorio
# ------------------------------
class RandomAgent:
    def get_action(self, game):
        player = game.current_player
        legal = game.legal_actions(player)
        return random.choice(legal) if legal else None

# ------------------------------
# Agente heurístico CORREGIDO
# ------------------------------
class HeuristicAgent:
    def get_action(self, game):
        player = game.current_player
        legal = game.legal_actions(player)
        if not legal:
            return None

        trick_cards = game.trick['cards']
        trump = game.trump

        # Primera carta de la baza: jugar la más baja
        if not trick_cards:
            return min(legal, key=lambda x: x[1])

        lead_suit = game.trick['suit']
        # Valor de la mejor carta del palo lead_suit ya jugada
        best_lead_value = 0
        for (_, card) in trick_cards:
            if card[0] == lead_suit:
                best_lead_value = max(best_lead_value, card[1])

        same_suit = [c for c in legal if c[0] == lead_suit]
        if same_suit:
            better = [c for c in same_suit if c[1] > best_lead_value]
            if better:
                return max(better, key=lambda x: x[1])   # la más alta que supere
            else:
                return min(same_suit, key=lambda x: x[1]) # la más baja
        else:
            trump_cards = [c for c in legal if c[0] == trump]
            if trump_cards:
                return min(trump_cards, key=lambda x: x[1])   # triunfo más bajo
            else:
                return min(legal, key=lambda x: x[1])         # cualquier carta baja

# ------------------------------
# Función para jugar una partida completa
# ------------------------------
def play_match(agent0, agent1, agent2, agent3, num_games=100):
    wins = [0,0]  # equipo 0, equipo 1
    for g in range(num_games):
        game = BriscasGame()
        game.deal(seed=g)
        while not game.is_terminal():
            player = game.current_player
            if player == 0:
                action = agent0.get_action(game)
            elif player == 1:
                action = agent1.get_action(game)
            elif player == 2:
                action = agent2.get_action(game)
            else:
                action = agent3.get_action(game)
            if action is None:
                break
            game.play_card(player, action)
        winner = game.get_winner()
        if winner != -1:
            wins[winner] += 1
    return wins[0]/num_games, wins[1]/num_games

# ------------------------------
# Ejecución de experimentos
# ------------------------------
if __name__ == "__main__":
    env = BriscasGame()
    mc_agent0 = MonteCarloAgent(env, 0, num_samples=50)
    mc_agent2 = MonteCarloAgent(env, 2, num_samples=50)
    random_agent = RandomAgent()
    heuristic_agent = HeuristicAgent()

    print("(MC+MC) vs (Random+Random)")
    wr0, wr1 = play_match(mc_agent0, random_agent, mc_agent2, random_agent, num_games=200)
    print(f"Equipo0 (MC) win rate: {wr0:.3f}")

    print("\n(MC+Heuristic) vs (Heuristic+Heuristic)")
    wr0, wr1 = play_match(mc_agent0, heuristic_agent, heuristic_agent, heuristic_agent, num_games=200)
    print(f"Equipo0 (MC+Heur) win rate: {wr0:.3f}")

    print("\n(MC+MC) vs (Heuristic+Heuristic)")
    wr0, wr1 = play_match(mc_agent0, heuristic_agent, mc_agent2, heuristic_agent, num_games=200)
    print(f"Equipo0 (MC) win rate: {wr0:.3f}")

(MC+MC) vs (Random+Random)
Equipo0 (MC) win rate: 0.775

(MC+Heuristic) vs (Heuristic+Heuristic)
Equipo0 (MC+Heur) win rate: 0.580

(MC+MC) vs (Heuristic+Heuristic)
Equipo0 (MC) win rate: 0.665
